<span STYLE="font-size:150%"> 
    Slicer 3D visualization 
</span>

Docker image: gnasello/slicer-env:2024-05-08.2 \
Latest update: 8 May 2024

- load .stl surface model files obtained from segmentations
- visualize models in 3D

Load libraries

In [ ]:
import slicer
import pyslicer as ps
from pathlib import Path
import pandas as pd

In [ ]:
# this cell is tagged 'parameters'
model_dir = './'
output_dir_path = 'segmented_volumes/ROI'
model_file_path = 'segmented_volumes/Bone.stl'

# Visualize model

Make ```segmented_volumes``` folder

In [ ]:
output_directory = Path(output_dir_path)

output_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# Switch to "One 3D view" layout
layoutManager = slicer.app.layoutManager()
layoutManager.setLayout(slicer.vtkMRMLLayoutNode.SlicerLayoutOneUp3DView)

slicer.mrmlScene.Clear()
ps.view.default_dark_3D_view()

model_dir_path = Path(model_dir)

model_file = model_dir_path / model_file_path

model = ps.model.load(model_file, color=(0.9450980392156862, 0.8392156862745098, 0.5686274509803921)) # "Bone" color in Slicer


In [ ]:
ps.view.reset_camera_to_fit_all_visible_models()

In [ ]:
camera_file = model_dir_path / 'segmented_volumes/camera_view.csv'

if camera_file.exists():
    
    df_camera = pd.read_csv(camera_file)

    ps.view.set_camera_3Dview(position=df_camera['position'],
                              viewAngle=df_camera['viewAngle'][0],
                              viewUp=df_camera['viewUp'],
                              parallelScale=df_camera['parallelScale'][0])

else:
    print("camera_view.csv file does not exist.")

# Optional

Adjust the camera manually

In [ ]:
ps.view.get_camera_3Dview(save_csv=True, csv_path=camera_file)

# Draw Region of Interest in Slicer UI

In [ ]:
f_output = 'ROI.mrk.json'

roiFilePath = output_directory / f_output

In [ ]:
if not roiFilePath.exists():

    # --- PARAMETERS ---
    roiSizeMm = [4, 2, 2]  # ROI size in mm
    
    # Get bounds in RAS
    bounds = [0]*6
    model.GetRASBounds(bounds)
    centerRAS = [
        (bounds[0] + bounds[1]) / 2,
        (bounds[2] + bounds[3]) / 2,
        (bounds[4] + bounds[5]) / 2
    ]
    
    # --- CREATE ROI NODE ---
    roiNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLMarkupsROINode", "ROI")
    roiNode.SetSize(roiSizeMm)
    roiNode.SetCenter(centerRAS)
    
    # Assume roiNode already exists.
    displayNode = roiNode.GetDisplayNode()
    
    # Enable interaction handles
    displayNode.SetHandlesInteractive(True)
    
    # Turn ON translation & rotation
    displayNode.SetTranslationHandleVisibility(True)
    displayNode.SetRotationHandleVisibility(True)
    
    # Turn OFF scaling
    displayNode.SetScaleHandleVisibility(False)

    print(f"Created new ROI node '{roiNode.GetName()}' with size {roiSizeMm} at center {centerRAS}.")
    print("ROI interaction updated: Translate + Rotate enabled, Scale disabled.")

else:

    roiNode = slicer.util.loadNodeFromFile(roiFilePath)
    print(f"Loaded ROI node from file: {roiFilePath}")

<span style="color:red; font-size:200%">Manual Task</span>

Before running the code below, go to the Slicer UI to modify the **ROI** position.

In [ ]:
centerRAS = [0, 0, 0]
roiNode.GetCenter(centerRAS)

size = roiNode.GetSize()

print("Current ROI center (RAS):", centerRAS)
print("Current ROI size (mm):", size)

In [ ]:
f_output = 'ROI.mrk.json'

slicer.util.exportNode(roiNode, output_directory / f_output)

In [ ]:
#nodename = 'ROI'
#roiNode = slicer.util.getNode(nodename)

# Get cylinder inside ROI

In [ ]:
import slicer, vtk
import numpy as np

# Get size (L,W,H)
size = np.array(roiNode.GetSize())

# Center in world coords
center = np.zeros(3)
roiNode.GetNthControlPointPositionWorld(0, center)

# Orientation matrix (local-to-world for ROI axes)
orientationMatrix = roiNode.GetObjectToNodeMatrix()
orientarion_array = np.array([[orientationMatrix.GetElement(i, j) for j in range(4)] for i in range(4)])

In [ ]:
import trimesh

box = trimesh.creation.box(extents=size, transform=orientarion_array)

# Compute minimum enclosing cylinder
cylinder_primitive = trimesh.bounds.minimum_cylinder(box)

# Get bounding_cylinder as mesh object
cylinder_mesh = box.bounding_cylinder

In [ ]:
cylinder_height = 3.5 #mm

# Create a model in 3D Slicer with the bounding cylinder cylinder
enclosing_cylinder_model = ps.model.create_hollow_cylinder(height=cylinder_height, 
                                                           radius_inner=0, radius_outer=cylinder_primitive['radius'], space =5, 
                                                           center=cylinder_mesh.center_mass,
                                                           direction=cylinder_mesh.direction.tolist(),
                                                           transform=False,
                                                           nameModel='FemurCylinder', 
                                                           #color=(128/255, 174/255, 128/255), 
                                                           color=(216/255, 101/255, 79/255),                                                            
                                                           opacity=0.4)

In [ ]:
large_radius = cylinder_primitive['radius']*1.5

# Create a model in 3D Slicer with the bounding cylinder cylinder
large_cylinder_model = ps.model.create_hollow_cylinder(height=cylinder_height, 
                                                           radius_inner=0, radius_outer=large_radius, space =5, 
                                                           center=cylinder_mesh.center_mass,
                                                           direction=cylinder_mesh.direction.tolist(),
                                                           transform=False,
                                                           nameModel='CallusCylinder', 
                                                           color=(128/255, 174/255, 128/255), 
                                                           opacity=0.4)

In [ ]:
# Create a model in 3D Slicer with the bounding cylinder cylinder
outside_cylinder_model = ps.model.create_hollow_cylinder(height=cylinder_height, 
                                                           radius_inner=cylinder_primitive['radius'], radius_outer=large_radius, space =5, 
                                                           center=cylinder_mesh.center_mass,
                                                           direction=cylinder_mesh.direction.tolist(),
                                                           transform=False,
                                                           nameModel='OutsideCylinder', 
                                                           color=(128/255, 174/255, 128/255), 
                                                           opacity=0.4)

# Export cylinders

In [ ]:
model = enclosing_cylinder_model

filename_output = model.GetName() + '.stl'

slicer.util.exportNode(model, output_directory / filename_output)

In [ ]:
model = large_cylinder_model

filename_output = model.GetName() + '.stl'

slicer.util.exportNode(model, output_directory / filename_output)

In [ ]:
model = outside_cylinder_model

filename_output = model.GetName() + '.stl'

slicer.util.exportNode(model, output_directory / filename_output)